# CareLoop Clinics: Part 3, predicting no-shows

Goal: a risk score the front desk can use the evening before an appointment to decide whom to phone.
Only about 1 booking in 5 is a no-show, so every model is judged by recall, precision and ROC-AUC as well as accuracy.

**Rule for the whole notebook:** the test set is for reporting only. It is never used to choose a model, a setting or a threshold.

Before running: upload `appointments_clean.csv` (the file made in Part 1) in the first cell. Do not shuffle or re-sort its rows.

In [ ]:
# Block 0: imports and file upload
import numpy as np
import pandas as pd
from google.colab import files

from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     cross_val_score, cross_val_predict)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

# Upload appointments_clean.csv (made in Part 1)
uploaded = files.upload()

## Task 3.1: Features, split and baseline

In [ ]:
# Task 3.1: build X and y, split, and find the "do nothing" baseline

df = pd.read_csv("appointments_clean.csv")   # important: don't shuffle or sort this
print("Loaded:", df.shape)

# reminder_sent is Yes/No text, so turn it into 1/0
df["reminder_sent"] = df["reminder_sent"].map({"Yes": 1, "No": 0})

# Keep only the columns the brief allows. IDs, dates, clinic_id, gender and status
# are left out on purpose (status IS the answer, so using it would be cheating).
num_cols = ["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows", "reminder_sent"]
X = pd.get_dummies(df[num_cols + ["booking_channel", "visit_type"]],
                   columns=["booking_channel", "visit_type"], dtype=int)
y = df["no_show"]

print("Number of feature columns:", X.shape[1])   # expect 11
print(list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Train rows:", len(X_train), "| Test rows:", len(X_test))   # expect 792 and 198
print("No-shows in test set:", y_test.sum())                      # expect 49

# Baseline: a lazy model that always says "Attended" (0)
baseline_acc = (y_test == 0).mean()
print("Baseline accuracy:", round(baseline_acc, 4))               # expect 0.7525 (149 of 198)

**What this shows:** there are 11 feature columns, 792 training rows and 198 test rows, with 49 no-shows in the test set. A model that always predicts "Attended" would already be right 75.25% of the time, so any real model has to beat that number, and accuracy alone is not enough to judge it.

## Task 3.2: Logistic regression

In [ ]:
# Task 3.2: logistic regression on SCALED features

# The scaler learns its mean/std from the training set only,
# then we apply the same numbers to the test set (no peeking).
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_s, y_train)

lr_pred = log_reg.predict(X_test_s)
lr_prob = log_reg.predict_proba(X_test_s)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, lr_pred), 3))   # about 0.788
print("Precision:", round(precision_score(y_test, lr_pred), 3))  # about 0.63
print("Recall   :", round(recall_score(y_test, lr_pred), 3))     # about 0.35
print("F1       :", round(f1_score(y_test, lr_pred), 3))         # about 0.45
print("ROC-AUC  :", round(roc_auc_score(y_test, lr_prob), 3))    # about 0.789
print("Confusion matrix:\n", confusion_matrix(y_test, lr_pred))  # about [[139, 10], [32, 17]]

# Which feature pushes the no-show risk up or down?
coef_table = pd.DataFrame({"feature": X.columns, "coefficient": log_reg.coef_[0]})
coef_table["exp(coef)"] = np.exp(coef_table["coefficient"])
print(coef_table.sort_values("coefficient").to_string(index=False))
# reminder_sent should be the most negative; lead_days and prior_no_shows clearly positive

# The coefficient is "per 1 standard deviation" because the features were scaled.
# To get the effect of a real No -> Yes change (0 to 1), divide by that feature's std first.
i = list(X.columns).index("reminder_sent")
coef_scaled = log_reg.coef_[0][i]
coef_real = coef_scaled / scaler.scale_[i]

print("\nreminder_sent, per 1 std (scaled):", round(np.exp(coef_scaled), 2))      # about 0.56
print("reminder_sent, No -> Yes (real)    :", round(np.exp(coef_real), 2))        # about 0.31
print("scaler std for reminder_sent       :", round(scaler.scale_[i], 2))         # about 0.49

**What logistic regression tells us**

The model gets about 79% accuracy and a ROC-AUC of about 0.79. It is careful rather than bold: when it flags a booking it is right roughly 63% of the time, but it catches only about 35% of the real no-shows.

The coefficients say that bookings made far ahead (`lead_days`) and patients with earlier no-shows (`prior_no_shows`) carry higher risk, while `reminder_sent` has the most negative coefficient.

The two reminder numbers (0.56 and 0.31) differ because they answer different questions. 0.56 is the effect of a one-standard-deviation change in the scaled feature, but `reminder_sent` is just 0/1 with a standard deviation of about 0.49, so that is not a real change. Dividing the coefficient by that standard deviation gives the effect of going from No to Yes, which is 0.31.

In this model, holding the other features constant, the odds of a no-show are about 69% lower for appointments that had a reminder. This is an association only. The data is observational, so it does not prove that sending reminders causes fewer no-shows.

## Task 3.3: Decision tree and overfitting

In [ ]:
# Task 3.3: same tree twice, once unlimited and once with a depth limit.
# Trees don't need scaling, so we use the original X_train / X_test.

tree_full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree_3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

for name, model in [("Full-depth tree", tree_full), ("Depth-3 tree", tree_3)]:
    tr = accuracy_score(y_train, model.predict(X_train))
    te = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: train = {tr:.2f}, test = {te:.2f}")

print("Full tree depth:", tree_full.get_depth())   # about 15
# Expect: full tree train 1.00, test about 0.67 (below the 0.7525 baseline!)
#         depth-3 tree train about 0.82, test about 0.78

**Overfitting in the decision tree**

The tree with no depth limit scores 1.00 on the training data but only about 0.67 on the test data, which is even below the 0.7525 "always predict Attended" baseline. It simply memorised the training rows, noise included. Limiting the depth to 3 brings training accuracy down to about 0.82 and lifts test accuracy to about 0.78, so the gap almost disappears.

## Task 3.4: Random forest and a comparison table

In [ ]:
# Task 3.4: random forest + side-by-side comparison

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("RF train accuracy:", round(accuracy_score(y_train, rf.predict(X_train)), 2))  # about 1.00
print("RF test accuracy :", round(accuracy_score(y_test, rf_pred), 2))               # about 0.75
print("RF recall        :", round(recall_score(y_test, rf_pred), 2))                 # about 0.31
print("RF ROC-AUC       :", round(roc_auc_score(y_test, rf_prob), 2))                # about 0.71

# Top 3 features the forest relied on
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 3 features:\n", importances.head(3).round(2))
# lead_days (~0.22), age (~0.20), distance_km (~0.19); order may vary

# Comparison table (for REPORTING only; the real model choice happens in Task 3.5)
tree3_pred = tree_3.predict(X_test)
tree3_prob = tree_3.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "model": ["Logistic regression", "Decision tree (depth 3)", "Random forest"],
    "accuracy": [accuracy_score(y_test, lr_pred), accuracy_score(y_test, tree3_pred), accuracy_score(y_test, rf_pred)],
    "recall":   [recall_score(y_test, lr_pred), recall_score(y_test, tree3_pred), recall_score(y_test, rf_pred)],
    "roc_auc":  [roc_auc_score(y_test, lr_prob), roc_auc_score(y_test, tree3_prob), roc_auc_score(y_test, rf_prob)],
}).round(2)
print("\n", comparison.to_string(index=False))
# Expect ROC-AUC: logistic about 0.77-0.79, tree about 0.68, forest about 0.71-0.75

**Comparing the three models**

On the test set, logistic regression has the highest ROC-AUC (about 0.79, against about 0.68 for the depth-3 tree and about 0.71 for the forest), so it separates risky bookings from safe ones best. The random forest memorises the training data (training accuracy 1.00) and ends up no better than the baseline on test accuracy, with low recall of about 0.31. Its top three features are `lead_days`, `age` and `distance_km`.

This table is only for reporting. The model is chosen from training data in Task 3.5, not from the test set.

## Task 3.5: Tune, choose the model and the threshold, then test once

In [ ]:
# Task 3.5 (a): tune the random forest with 5-fold CV on the TRAINING set only

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
    cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)               # any grid value is fine
print("Best CV ROC-AUC:", round(grid.best_score_, 2))  # about 0.77

# Task 3.5 (b): pick the final model WITHOUT touching the test set.
# Logistic regression goes in a Pipeline so the scaler is re-fitted inside every CV fold
# (scaling the whole training set first would leak information between folds).
lr_pipe = Pipeline([("scaler", StandardScaler()),
                    ("model", LogisticRegression(max_iter=1000))])

cv_scores = {
    "Logistic regression": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision tree (depth 3)": cross_val_score(tree_3, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest (100 trees)": cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Tuned random forest": grid.best_score_,
}
for name, score in cv_scores.items():
    print(f"{name}: CV ROC-AUC = {score:.2f}")
# Expect: logistic about 0.77, tree about 0.73, forest about 0.75, tuned forest about 0.77

models_map = {
    "Logistic regression": lr_pipe,
    "Decision tree (depth 3)": tree_3,
    "Random forest (100 trees)": rf,
    "Tuned random forest": grid.best_estimator_,
}
winner_name = max(cv_scores, key=cv_scores.get)
winner_model = models_map[winner_name]
print("\nWinner:", winner_name)    # expect Logistic regression

In [ ]:
# Task 3.5 (c): choose the threshold using out-of-fold predictions on the TRAINING set.
# Each training row gets a probability from a model that never saw that row.
# (The test set is still untouched here.)

oof = cross_val_predict(winner_model, X_train, y_train, cv=5, method="predict_proba")[:, 1]

rows = []
for t in [0.5, 0.35]:
    flagged = (oof >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_train, flagged),
        "recall": recall_score(y_train, flagged),
        "f1": f1_score(y_train, flagged),
        "flagged": int(flagged.sum()),
    })
thr_table = pd.DataFrame(rows).round(2)
print(thr_table.to_string(index=False))
# Expect: 0.5  -> precision about 0.58, recall about 0.25, F1 about 0.35, about 73 flagged
#         0.35 -> precision about 0.51, recall about 0.47, F1 about 0.49, about 154 flagged

# The threshold with the higher F1 wins
chosen_t = thr_table.loc[thr_table["f1"].idxmax(), "threshold"]
print("\nChosen threshold:", chosen_t)    # expect 0.35

**Choosing the model and the threshold**

Using 5-fold cross-validation on the training set only, logistic regression has the highest ROC-AUC (about 0.77), about level with the tuned random forest and ahead of the plain forest and the depth-3 tree. It is also the simplest of the good options, so it is the final model.

For the threshold I compared 0.5 and 0.35 on out-of-fold training predictions. At 0.5 the model flags about 73 bookings with recall of only about 0.25 (F1 about 0.35). At 0.35 it flags about 154 with recall of about 0.47 (F1 about 0.49). The higher F1 is at 0.35, so that is the threshold.

Recall matters more here because a missed no-show means an empty doctor slot, while an extra reminder call costs only about a minute of the front desk's time.

In [ ]:
# Task 3.5 (d) and (e): the one and only test-set evaluation of the winning model

winner_model.fit(X_train, y_train)
probs = winner_model.predict_proba(X_test)[:, 1]
flag_test = (probs >= chosen_t).astype(int)

print("Test precision:", round(precision_score(y_test, flag_test), 2))   # about 0.57
print("Test recall   :", round(recall_score(y_test, flag_test), 2))      # about 0.57
print("Number flagged:", int(flag_test.sum()))                           # about 49
print("Test ROC-AUC  :", round(roc_auc_score(y_test, probs), 2))         # about 0.79

# Save the risk scores for every test row.
# X_test keeps the original row numbers, so we can look up each appointment_id.
test_ids = df.loc[X_test.index, "appointment_id"].values

risk_out = pd.DataFrame({
    "appointment_id": test_ids,
    "no_show": y_test.values,
    "risk_prob": probs,
    "flag": flag_test,
})
risk_out.to_csv("test_risk_scores.csv", index=False)

print("\nRows saved:", len(risk_out))                        # expect 198
print("Rows with flag = 1:", int(risk_out["flag"].sum()))    # must equal 'Number flagged' above
files.download("test_risk_scores.csv")

**Final test result (evaluated once)**

With logistic regression at a threshold of 0.35, the test set gives precision of about 0.57 and recall of about 0.57, with about 49 bookings flagged and a ROC-AUC of about 0.79. The scores for all 198 test rows are saved in `test_risk_scores.csv`, and the number of rows with `flag = 1` equals the number flagged above.

In practice a clinic would choose its threshold by how many follow-up calls the front desk can really make in an evening, weighed against what an empty slot costs. A busy desk with few calls should use a higher threshold, and a clinic with spare time can lower it and call more patients.

**Honest limits of this analysis**
- The medians and outlier fences in Part 1 were calculated on all 990 rows before the split, so the test rows had a small indirect influence on the cleaning.
- The same patient can appear in several bookings, so a random split can put one patient in both train and test.
- The split is random, whereas a real system would be trained on the past and used on future bookings.